# Detecting massive black hole mergers in LISA data with a CNN
**Class project: recreation and extension of Houba et al. (2024), "Detection and Prediction of Future Massive Black Hole Mergers with Machine Learning and Truncated Waveforms"**

**What this notebook does**
1. Loads the LISA Spritz files (`mbhb1`, `mbhb2`, `vgb`) and builds the TDI **A** and **E** channels from X, Y, Z.
2. Turns 10-day windows of data into log-frequency spectrogram images (the paper's CNN input).
3. Simulates a training set: realistic noise (shaped to the real data's PSD) plus injected inspiral chirps.
4. Trains the paper's CNN twice: **A only** (reproduction) and **A + E** (our extension).
5. Compares the two on simulated test data and on the real Spritz mergers (`mbhb1`, `mbhb2`).

**Simplifications vs. the paper** (state these in your presentation)
- No reinforcement learning or merger-time prediction. This is the detection CNN only.
- Training signals are leading-order (Newtonian) chirps, not IMRPhenomD plus the full LISA response, and the training set is thousands of events instead of 250,000.
- Detection is a **binary** output (merger present or not) instead of the paper's "vertical pixel position" regression.

## 1. Setup and file access

In [ ]:
%pip install -q h5py scikit-learn
# Local Mac only, if TensorFlow is missing:  %pip install tensorflow

In [ ]:
import os, sys, glob, time
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.signal import spectrogram, welch
from scipy.interpolate import interp1d
from sklearn.metrics import roc_auc_score, roc_curve
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IN_COLAB = "google.colab" in sys.modules

# ---- WHERE YOUR FILES LIVE -------------------------------------------------
# Running locally on your Mac: files are read straight from the Desktop.
LOCAL_DIR = "/Users/jordanmasterson/Desktop/NN PS/Project"
# Running in Google Colab: Colab cannot see your Desktop. Upload the folder
# "NN PS" to Google Drive (My Drive), then this path will work after mounting.
DRIVE_DIR = "/content/drive/MyDrive/NN PS/Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DRIVE_DIR
else:
    DATA_DIR = LOCAL_DIR

FILES = {
    "mbhb1": "LDC2_spritz_mbhb1_training_v2.h5",
    "mbhb2": "LDC2_spritz_mbhb2_training_v2.h5",
    "vgb":   "LDC2_spritz_vgb_training_v2.h5",
    "orbits": "orbits.h5",
}

def find_file(name):
    # exact match in DATA_DIR first, then any file ending in the same name
    p = os.path.join(DATA_DIR, name)
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join(DATA_DIR, "*" + name))
    return hits[0] if hits else None

PATHS = {k: find_file(v) for k, v in FILES.items()}

# Colab fallback: if the Drive folder was not found, upload the files by hand
if IN_COLAB and not PATHS["mbhb1"]:
    from google.colab import files
    print("Files not found on Drive. Select the .h5 files to upload.")
    files.upload()
    DATA_DIR = "/content"
    PATHS = {k: find_file(v) for k, v in FILES.items()}

for k, p in PATHS.items():
    print(f"{k:7s} ->", p if p else "NOT FOUND")
assert PATHS["mbhb1"] and PATHS["mbhb2"], "mbhb1 / mbhb2 are required. Check DATA_DIR above."

## 2. Look inside the files

In [ ]:
def show_structure(path, max_items=12):
    with h5py.File(path, "r") as h:
        print(os.path.basename(path), "| top-level groups:", list(h.keys()))
        for g in ["obs", "clean", "noisefree", "gal", "sky"]:
            if g in h and "tdi" in h[g]:
                d = h[g]["tdi"]
                print(f"  {g}/tdi: shape {d.shape}, fields {d.dtype.names}")

for k in ["mbhb1", "mbhb2", "vgb"]:
    if PATHS[k]:
        show_structure(PATHS[k])

def merger_info(path):
    with h5py.File(path, "r") as h:
        c = h["sky/cat"][()]
    return {n: float(np.ravel(c[n])[0]) for n in ["Mass1", "Mass2", "CoalescenceTime"]}

for k in ["mbhb1", "mbhb2"]:
    print(k, merger_info(PATHS[k]))

## 3. Load the data and build TDI A and E
The files store the Michelson channels X, Y, Z. The orthogonal channels are
**A = (Z − X)/√2**, **E = (X − 2Y + Z)/√6** (and T = (X+Y+Z)/√3, not used here).
The `obs` data contain NaN gaps, which we fill by linear interpolation.

In [ ]:
FS = 0.2                      # Hz  (5 s cadence in the Spritz files)
DT = 1 / FS
WIN_DAYS = 10
WIN_N = int(WIN_DAYS * 86400 * FS)    # 172,800 samples per 10-day window

def fill_gaps(x, gap):
    if not gap.any():
        return x
    idx = np.arange(len(x))
    out = x.copy()
    out[gap] = np.interp(idx[gap], idx[~gap], x[~gap])
    return out

def load_tdi(path, group="obs"):
    with h5py.File(path, "r") as h:
        d = h[f"{group}/tdi"][:, 0]
    t = d["t"].astype(np.float64)
    X, Y, Z = (d[c].astype(np.float64) for c in "XYZ")
    gap = np.isnan(X) | np.isnan(Y) | np.isnan(Z)
    X, Y, Z = (fill_gaps(np.where(gap, 0.0, v), gap) for v in (X, Y, Z))
    A = (Z - X) / np.sqrt(2)
    E = (X - 2 * Y + Z) / np.sqrt(6)
    return t, A, E, gap

REAL = {}
for k in ["mbhb1", "mbhb2"]:
    t, A, E, gap = load_tdi(PATHS[k])
    REAL[k] = dict(t=t, A=A, E=E, gap=gap, **{"tm": merger_info(PATHS[k])["CoalescenceTime"]})
    print(f"{k}: {len(t)} samples, {gap.sum()} gap samples, merger {(REAL[k]['tm']-t[0])/86400:.2f} d after start")

## 4. From time series to an image (CNN input)
Each 10-day window becomes a 64×64 image: time on x, **log-spaced** frequency on y (1e-4 to 5e-2 Hz).
We take the log-power, divide out each frequency row's median (so the colored noise floor is removed and a chirp stands out), then scale.

In [ ]:
IMG = 64
NPERSEG, NOVERLAP = 2048, 1536
F_LO, F_HI = 1e-4, 5e-2
FREQ_TARGET = np.logspace(np.log10(F_LO), np.log10(F_HI), IMG)

def to_image(x):
    f, t, S = spectrogram(x, fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP,
                          detrend="linear", scaling="density")
    S = interp1d(f, S, axis=0)(FREQ_TARGET)                  # log-frequency axis
    S = np.log10(S / np.median(S) + 1e-6)
    cols = np.array_split(np.arange(S.shape[1]), IMG)         # time -> 64 columns
    S = np.stack([S[:, c].mean(1) for c in cols], axis=1)
    S -= np.median(S, axis=1, keepdims=True)                  # whiten each frequency row
    S /= (S.std() + 1e-9)
    return S[::-1].astype(np.float32)                         # low frequency at the bottom

def make_input(A_win, E_win):
    return np.stack([to_image(A_win), to_image(E_win)], axis=-1)   # (64, 64, 2)

# Real data example: last 10 days before the mbhb1 merger (ends 1 hour before)
r = REAL["mbhb1"]
end = np.searchsorted(r["t"], r["tm"] - 3600)
img = make_input(r["A"][end - WIN_N:end], r["E"][end - WIN_N:end])
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for i, name in enumerate(["A", "E"]):
    ax[i].imshow(img[..., i], aspect="auto", cmap="gray_r",
                 extent=[-10, 0, np.log10(F_LO), np.log10(F_HI)])
    ax[i].set_title(f"mbhb1, TDI {name}, last 10 days"); ax[i].set_xlabel("days to merger"); ax[i].set_ylabel("log10 f [Hz]")
plt.tight_layout(); plt.show()

## 5. Simulating training data
**Noise:** we estimate the real noise PSD of A and E (from `vgb` if available, since it has no merger; otherwise from the first 10 days of the two `mbhb` files) and generate new Gaussian noise with that spectrum.

**Signal:** a leading-order inspiral chirp
$f(\tau)=\frac{1}{\pi}\left(\frac{5}{256\,\tau}\right)^{3/8}\left(\frac{G\mathcal{M}}{c^3}\right)^{-5/8}$, with phase $\Phi(\tau)=-2\left(5\frac{G\mathcal{M}}{c^3}\right)^{-5/8}\tau^{5/8}$, where $\tau$ is the time to merger.
Amplitude grows as $f^{2/3}$ and is scaled relative to the noise at that frequency. E gets the same chirp with a random amplitude ratio and phase offset, which stands in for the sky-dependent A/E response. This is why E can add information beyond A.

In [ ]:
def estimate_psd():
    sources = []
    if PATHS.get("vgb"):
        try:
            t, A, E, g = load_tdi(PATHS["vgb"])
            sources.append((A, E)); print("PSD source: vgb (whole file)")
        except Exception as e:
            print("Could not read vgb:", e)
    if not sources:
        for k in ["mbhb1", "mbhb2"]:
            sources.append((REAL[k]["A"][:WIN_N], REAL[k]["E"][:WIN_N]))
        print("PSD source: first 10 days of mbhb1 and mbhb2")
    SA, SE = [], []
    for A, E in sources:
        f, sa = welch(A, fs=FS, nperseg=8192, average="median")
        _, se = welch(E, fs=FS, nperseg=8192, average="median")
        SA.append(sa); SE.append(se)
    return f, np.mean(SA, axis=0), np.mean(SE, axis=0)

F_PSD, S_A, S_E = estimate_psd()
plt.loglog(F_PSD[1:], np.sqrt(S_A[1:]), label="A"); plt.loglog(F_PSD[1:], np.sqrt(S_E[1:]), label="E")
plt.xlabel("Hz"); plt.ylabel("ASD"); plt.legend(); plt.title("Estimated noise ASD"); plt.show()

def make_noise(S, n, rng):
    w = rng.standard_normal(n)
    freqs = np.fft.rfftfreq(n, d=DT)
    amp = np.sqrt(np.interp(freqs, F_PSD, S) * FS / 2)
    amp[0] = 0.0
    return np.fft.irfft(np.fft.rfft(w) * amp, n)

In [ ]:
G_MSUN = 4.925491e-6          # G*Msun/c^3 in seconds
DF_SPEC = FS / NPERSEG        # spectrogram frequency resolution

# ---- knobs you can tune --------------------------------------------------
AMP_RANGE  = (3.0, 60.0)      # log-uniform "loudness" a; larger = easier detection
TAU_RANGE_D = (1/24, 20.0)    # window ends between 1 hour and 20 days before merger
MASS_RANGE = (1e5, 1e7)       # total mass [Msun]
Q_RANGE    = (0.1, 1.0)
# --------------------------------------------------------------------------

def chirp(tau_s, Mc_s):
    f = (1 / np.pi) * (5 / (256 * tau_s)) ** 0.375 * Mc_s ** (-0.625)
    phi = -2 * (5 * Mc_s) ** (-0.625) * tau_s ** 0.625
    return f, phi

def logu(rng, lo, hi):
    return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))

def simulate(rng, positive):
    nA, nE = make_noise(S_A, WIN_N, rng), make_noise(S_E, WIN_N, rng)
    tau_end = logu(rng, *TAU_RANGE_D)
    if positive:
        M = logu(rng, *MASS_RANGE); q = rng.uniform(*Q_RANGE)
        eta = q / (1 + q) ** 2
        Mc_s = M * eta ** 0.6 * G_MSUN
        f_isco = 1 / (6 ** 1.5 * np.pi * M * G_MSUN)
        tau = tau_end * 86400 + (WIN_N - 1 - np.arange(WIN_N)) * DT      # time to merger per sample
        f, phi = chirp(tau, Mc_s)
        a = logu(rng, *AMP_RANGE)
        env = (f < f_isco).astype(float)                                  # inspiral only
        phase0 = rng.uniform(0, 2 * np.pi)
        rE, dphE = rng.uniform(0.3, 1.0), rng.uniform(0, 2 * np.pi)
        gf = (f / 1e-3) ** (2 / 3)
        ampA = a * np.sqrt(np.interp(f, F_PSD, S_A) * DF_SPEC) * gf * env
        ampE = rE * a * np.sqrt(np.interp(f, F_PSD, S_E) * DF_SPEC) * gf * env
        nA = nA + ampA * np.cos(phi + phase0)
        nE = nE + ampE * np.cos(phi + phase0 + dphE)
    return make_input(nA, nE), int(positive), (tau_end if positive else np.nan)

# Visual sanity check
rng = np.random.default_rng(0)
fig, ax = plt.subplots(2, 4, figsize=(14, 6))
for j, (pos, tau) in enumerate([(True, None)] * 2 + [(False, None)] * 2):
    x, y, tau_end = simulate(rng, pos)
    for i, nm in enumerate(["A", "E"]):
        ax[i, j].imshow(x[..., i], aspect="auto", cmap="gray_r")
        ax[i, j].set_title(f"{nm}: {'merger' if y else 'noise only'}" + (f", tau_end={tau_end:.1f} d" if y else ""))
        ax[i, j].axis("off")
plt.tight_layout(); plt.show()

## 6. Generate the datasets

In [ ]:
N_TRAIN, N_VAL, N_TEST = 4000, 600, 1200     # total samples, half positive / half negative
SEED = 42

def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, IMG, IMG, 2), np.float32); y = np.zeros(n, np.float32); tau = np.full(n, np.nan)
    for i in range(n):
        pos = (i % 2 == 0)
        X[i], y[i], tau[i] = simulate(rng, pos)
    p = rng.permutation(n)
    return X[p], y[p], tau[p]

t0 = time.time()
X_tr, y_tr, _      = make_dataset(N_TRAIN, SEED)
X_va, y_va, _      = make_dataset(N_VAL, SEED + 1)
X_te, y_te, tau_te = make_dataset(N_TEST, SEED + 2)
print(f"Generated in {time.time()-t0:.0f} s | train {X_tr.shape}, val {X_va.shape}, test {X_te.shape}")

## 7. The model
The paper's CNN: two conv layers (32 filters), then four conv layers (64 filters) with 2×2 max-pooling after each pair, then dense layers of 64, 64, 32 and an output.
We use a sigmoid output (detection probability) because our label is binary. The only difference between the two experiments is the number of input channels.

In [ ]:
def build_model(in_channels):
    inp = keras.Input((IMG, IMG, in_channels))
    x = inp
    for filters in (32, 64, 64):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.Flatten()(x)
    for u in (64, 64, 32):
        x = layers.Dense(u, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m

EPOCHS, BATCH = 25, 64

def train(channels, name):
    tf.keras.utils.set_random_seed(SEED)
    m = build_model(len(channels))
    cb = [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss")]
    h = m.fit(X_tr[..., channels], y_tr, validation_data=(X_va[..., channels], y_va),
              epochs=EPOCHS, batch_size=BATCH, callbacks=cb, verbose=2)
    return m, h

print("=== Model A only ===")
model_A, hist_A = train([0], "A")
print("=== Model A + E ===")
model_AE, hist_AE = train([0, 1], "A+E")

for h, n in [(hist_A, "A only"), (hist_AE, "A + E")]:
    plt.plot(h.history["val_loss"], label=f"{n} val loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()

## 8. Results on simulated test data

In [ ]:
MODELS = {"A only": (model_A, [0]), "A + E": (model_AE, [0, 1])}
preds = {n: m.predict(X_te[..., ch], verbose=0).ravel() for n, (m, ch) in MODELS.items()}

for n, p in preds.items():
    acc = ((p > 0.5) == (y_te > 0.5)).mean()
    fpr = (p[y_te == 0] > 0.5).mean()
    print(f"{n:7s} accuracy {acc:.3f} | AUC {roc_auc_score(y_te, p):.3f} | false-alarm rate {fpr:.3f}")

# Detection rate vs. time to merger (the paper's Fig. 8 analogue)
bins = np.array([1/24, 0.25, 1, 2, 5, 10, 15, 20])
centers = np.sqrt(bins[:-1] * bins[1:])
plt.figure(figsize=(7, 4))
for n, p in preds.items():
    rate = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        sel = (y_te == 1) & (tau_te >= lo) & (tau_te < hi)
        rate.append((p[sel] > 0.5).mean() * 100 if sel.sum() else np.nan)
    plt.semilogx(centers, rate, "o-", label=n)
plt.gca().invert_xaxis(); plt.xlabel("window end: days before merger"); plt.ylabel("correct detections [%]")
plt.title("Detection rate vs. time to merger (simulated test set)"); plt.legend(); plt.grid(alpha=.3); plt.show()

plt.figure(figsize=(4.5, 4))
for n, p in preds.items():
    fp, tp, _ = roc_curve(y_te, p); plt.plot(fp, tp, label=n)
plt.plot([0, 1], [0, 1], "k--", lw=.5); plt.xlabel("false positive rate"); plt.ylabel("true positive rate"); plt.legend(); plt.title("ROC"); plt.show()

## 9. Real Spritz mergers (`mbhb1`, `mbhb2`)
We slide a 10-day window through each real file (stride 6 hours) and plot the detection probability against the time to the true merger. mbhb1 is the paper's validation event, where they report detection about 20 days before merger.

In [ ]:
STRIDE = int(6 * 3600 * FS)

def run_real(key):
    r = REAL[key]
    ends = np.arange(WIN_N, len(r["t"]) + 1, STRIDE)
    tau_d = (r["tm"] - r["t"][ends - 1]) / 86400
    keep = tau_d > 1/24                        # stop 1 hour before merger (inspiral only)
    ends, tau_d = ends[keep], tau_d[keep]
    X = np.stack([make_input(r["A"][e - WIN_N:e], r["E"][e - WIN_N:e]) for e in ends])
    return tau_d, {n: m.predict(X[..., ch], verbose=0).ravel() for n, (m, ch) in MODELS.items()}

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, key in zip(axs, ["mbhb1", "mbhb2"]):
    tau_d, P = run_real(key)
    for n, p in P.items():
        ax.plot(tau_d, p, "o-", ms=3, label=n)
    ax.axhline(0.5, color="k", ls="--", lw=.6); ax.invert_xaxis()
    ax.set_title(f"{key} (window end vs. true merger)"); ax.set_xlabel("days before merger"); ax.grid(alpha=.3)
axs[0].set_ylabel("detection probability"); axs[0].legend()
plt.tight_layout(); plt.show()

## 10. Notes for your write-up
- **Noise PSD leakage:** if `vgb` could not be read, the noise model was estimated from the first 10 days of the same `mbhb` files used for the real-data test. This only matches the noise level, not the signal, but mention it.
- **Why A+E might help:** the signal appears in both channels with sky-dependent amplitude and phase, while the noise is independent, so two views give extra evidence.
- **Tuning:** if both models score near 50%, raise `AMP_RANGE`. If both score near 100%, lower it until the detection-vs-time curve is informative.
- **Next steps:** add real merger-free windows from `vgb` as negatives, use IMRPhenomD waveforms with `lisagwresponse`, or restore the paper's regression label.